# CKAN API Documentation (Python + Jupyter)
## Introduction

This notebook documents how to interact with a CKAN instance using its HTTP API from Python.
It focuses on practical, copy/paste-friendly examples for common tasks:

Read-only (no auth required): search datasets, read dataset/org/group metadata, list resources, etc.

Write operations (auth required): create, update, and delete (CUD) datasets, resources, organizations, etc.

Each section will build up from basics (connection + GET requests) to authenticated workflows, with clear request/response examples.

## Requirements
Python libraries

You’ll need:

- ckanapi

- python-dotenv (optional, load environment variables from a .env file)

## CKAN access & credentials

- CKAN base URL (e.g., https://your-ckan-site.example)

- API key (required for Create/Update/Delete endpoints)

Generate or copy your API key from your CKAN user profile (or ask an admin).

Keep it secret (in `.env` file) and do not commit it into git.

Notes

Read-only endpoints usually work without an API key.

Authenticated requests typically pass the key via the Authorization header.

In [3]:
# !pip install ckanapi

In [1]:
from ckanapi import RemoteCKAN
import os
from dotenv import load_dotenv


# -----------------------------
# Configuration (env vars)
# -----------------------------
# Loads variables from a local .env file (if present)
# Example .env:
#   CKAN_BASE_URL=https://your-ckan-site.example
#   CKAN_API_KEY=xxxx-xxxx-xxxx-xxxx
load_dotenv()

CKAN_BASE_URL = os.getenv("CKAN_BASE_URL", "").rstrip("/")
CKAN_API_KEY = os.getenv("CKAN_API_KEY")  # required for Create/Update/Delete (CUD) actions

# Initialise the client

in the client initialisation you can also set the user-agent. `user_agent` is just the HTTP User-Agent header that ckanapi will send with every request to CKAN.

**What it’s for**

- Helps CKAN admins identify traffic in logs (e.g., “who is hammering the API?”).

- Useful for debugging, analytics, and sometimes rate-limiting / allow-listing.

- Some proxies/WAFs block requests with an empty or “unknown” user agent.

**Is it mandatory?**

No. Our CKAN site will accept requests without you explicitly setting it. But it’s strongly recommended to set a meaningful one.

In [2]:
ua = "pidinst-api-docs/0.1 (contact: you@org.example)"

rc = RemoteCKAN(CKAN_BASE_URL, apikey=CKAN_API_KEY, user_agent=ua)

# Search queries

### Simple search

In [3]:
rc.action.package_search()

{'count': 3,
 'facets': {},
 'results': [{'alternate_identifier_obj': '',
   'author': None,
   'author_email': None,
   'citation': 'Test 01 [2026-02-23]. ',
   'creator_user_id': '7a63c49b-b3aa-4706-830d-460ebd35490e',
   'credit': '',
   'date': '[{"date_type": "Commissioned", "date_value": "2020-01-10"}]',
   'description': 'test 01',
   'doi': '10.5555/ggz8duhk',
   'epsg': '4326 - WGS 84',
   'epsg_code': '4326',
   'funder': '',
   'gcmd_keywords': '',
   'gcmd_keywords_code': '',
   'id': '0b213d01-83dd-4dcb-b55c-b2f9bfc544b6',
   'instrument_type': '[{"instrument_type_identifier": "", "instrument_type_identifier_type": "", "instrument_type_name": "Microscope"}]',
   'isopen': False,
   'license_id': None,
   'license_title': None,
   'locality': '',
   'location_choice': 'point',
   'location_data': {'type': 'FeatureOrganisation',
    'features': [{'type': 'Feature',
      'geometry': {'type': 'Point',
       'coordinates': [115.87971826628973, -31.98555697278596]},
      'pro

### Solr Search

More detailed complex query syntax examples can be found in the [SOLR documentation](https://solr.apache.org/guide/6_6/common-query-parameters.html)

In [4]:
rc.action.package_search(q='title:"test" +version_number:2')

{'count': 1,
 'facets': {},
 'results': [{'alternate_identifier_obj': '',
   'author': None,
   'author_email': None,
   'citation': 'Test 01 [2026-02-10]. ',
   'creator_user_id': '7a63c49b-b3aa-4706-830d-460ebd35490e',
   'credit': '',
   'date': '[{"date_type": "Commissioned", "date_value": "2020-01-10"}]',
   'description': 'test 01',
   'doi': '10.5555/e8516to3',
   'epsg': '4326 - WGS 84',
   'epsg_code': '4326',
   'gcmd_keywords': '',
   'gcmd_keywords_code': '',
   'id': '2cd4c016-5294-4ad2-bd02-0ef62669f7ee',
   'instrument_type': '[{"instrument_type_identifier": "", "instrument_type_identifier_type": "", "instrument_type_name": "Microscope"}]',
   'isopen': False,
   'license_id': None,
   'license_title': None,
   'locality': '',
   'location_choice': 'point',
   'location_data': {'type': 'FeatureOrganisation',
    'features': [{'type': 'Feature',
      'geometry': {'type': 'Point',
       'coordinates': [115.87971826628973, -31.98555697278596]},
      'properties': {}}]},


# View Our Site's Dataset Schema

In [9]:
types =rc.action.scheming_dataset_schema_list()
schema = rc.action.scheming_dataset_schema_show(type=types[0])
schema

{'scheming_version': 2,
 'dataset_type': 'dataset',
 'about': 'AuScope Instrument Registry schema based on PIDINST',
 'about_url': 'https://www.github.com/AuScope/AuScope-instrument-registry',
 'dataset_fields': [{'field_name': 'doi',
   'label': 'Identifier',
   'form_placeholder': 'Persistent identifier (DOI) of the instrument instance',
   'form_attrs': {'disabled': True},
   'help_text': 'Unique string that identifies the instrument instance',
   'groupBy': 'About Instrument'},
  {'field_name': 'schema_version',
   'label': 'Schema Version',
   'form_snippet': 'hidden_field.html',
   'form_placeholder': 'Version number of the PIDINST schema used in this record',
   'help_text': 'Version number of the PIDINST schema used in this record',
   'groupBy': 'About Instrument'},
  {'validators': 'if_empty_same_as(name) unicode_safe',
   'form_snippet': 'large_text.html',
   'form_attrs': {'data-module': 'slug-preview-target'},
   'field_name': 'title',
   'label': 'Title',
   'preset': 'ti

In [12]:
actions = set(rc.action.)
print(actions)

CKANAPIError: ['http://localhost:5000/api/action/action_list', 400, '"Bad request - Action name not known: action_list"']